# SatQuery AI — Phase 1 zero-shot baselines

Runs the eval harness with the **untuned** `qwen2_5_vl` and the zero-shot RS
baseline `earthdial` over every benchmark whose files are present on this
machine, and renders the report. These are the "before" numbers of the
adaptation table: without them, the Phase 2 QLoRA run has nothing to prove.

Free-tier reality, encoded in `satquery.config` and not to be overridden here:

* **T4 (SM75) only** — fp16 + sdpa. bf16 and FlashAttention-2 both need SM80.
* On Kaggle pick the **2×T4** accelerator; Unsloth (Phase 2) will not run on a P100.
* Model id stays `Qwen/Qwen2.5-VL-7B-Instruct` (Apache-2.0). The 3B is the Qwen
  *Research* licence — non-commercial — so it is never substituted to save VRAM.

Every cell is a handful of lines on purpose. The logic lives in the `satquery`
package and is unit-tested on a laptop with numpy alone, so a broken baseline
run can be debugged without paying for a GPU session.

## 1. Extras

In [ ]:
# torch is preinstalled on Colab/Kaggle against their exact CUDA build --
# reinstalling it is the fastest way to lose the GPU for the session.
%pip install -q "transformers>=4.49" accelerate bitsandbytes qwen-vl-utils peft pillow tqdm
# Only needed if you feed real GeoTIFFs; the benchmarks below ship PNG/JPEG.
# %pip install -q rasterio

## 2. Repo

In [ ]:
import os, subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/YOUR_USER/SatQueryAI.git"  # or mount from Drive

_candidates = [Path("/kaggle/working/SatQueryAI"), Path("/content/SatQueryAI"), Path.cwd()]
REPO = next((p for p in _candidates if (p / "satquery").is_dir()), None)
if REPO is None:
    REPO = Path("/content" if Path("/content").is_dir() else ".") / "SatQueryAI"
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO)], check=True)

os.chdir(REPO)
sys.path.insert(0, str(REPO))  # run from source: no pip install -e on a read-only FS
print(REPO, "| python", sys.version.split()[0])

## 3. Where the benchmarks live

In [ ]:
# `SATQUERY_DATA_<NAME>` overrides one dataset, `SATQUERY_DATA_ROOT` the parent
# of the rest. Nothing in the package hardcodes a path, so attaching a Kaggle
# dataset under a different slug only changes this dict.
KAGGLE_INPUTS = {
    "rsvqa_lr": "/kaggle/input/rsvqa-lr",
    "rsvqa_hr": "/kaggle/input/rsvqa-hr",
    "vrsbench_vqa": "/kaggle/input/vrsbench",
    "vrsbench_caption": "/kaggle/input/vrsbench",
    "vrsbench_ground": "/kaggle/input/vrsbench",
    "dior_rsvg": "/kaggle/input/dior-rsvg",
    "cdvqa": "/kaggle/input/cdvqa",
    "levir_cc": "/kaggle/input/levir-cc",
    "levir_mci": "/kaggle/input/levir-mci",
    "bigearthnet_mm": "/kaggle/input/bigearthnet-mm",
}
for _name, _path in KAGGLE_INPUTS.items():
    if Path(_path).is_dir():
        os.environ[f"SATQUERY_DATA_{_name.upper()}"] = _path

os.environ.setdefault("SATQUERY_DATA_ROOT", str(REPO / "data"))
os.environ.setdefault("SATQUERY_RUNS_DIR", str(REPO / "runs"))
os.environ.setdefault("SATQUERY_MAX_SIDE", "768")  # the cap Phase 2 trains at
RUNS = Path(os.environ["SATQUERY_RUNS_DIR"])

## 4. Environment check

In [ ]:
CLI = [sys.executable, "-m", "satquery.cli"]


def sq(*args: str, check: bool = True) -> str:
    """Run the packaged CLI. The notebook never reimplements what it does."""
    proc = subprocess.run(CLI + [str(a) for a in args], text=True, capture_output=True)
    print(proc.stdout, proc.stderr, sep="")
    if check and proc.returncode:
        raise RuntimeError(f"satquery {' '.join(map(str, args))} -> exit {proc.returncode}")
    return proc.stdout


sq("doctor")
sq("datasets")

## 5. What can actually run

In [ ]:
from satquery.registry import DATASETS, build_dataset, build_predictor, task_of

PREDICTORS = ("qwen2_5_vl", "earthdial")
SUPPORTED = {p: set(build_predictor(p).supported) for p in PREDICTORS}  # no weights loaded


def available_benchmarks() -> list[tuple[str, str]]:
    found = []
    for name, (_target, task, split) in DATASETS.items():
        try:
            ds = build_dataset(name, split=split)
        except Exception as exc:  # adapter not written / bad split: report, do not abort
            print(f"  ??  {name:18s} {type(exc).__name__}: {exc}")
            continue
        ok = ds.available()
        print(f"  {'ok ' if ok else '-- '} {name:18s} {task:18s} {ds.root}")
        if ok:
            found.append((name, split))
    return found


BENCHMARKS = available_benchmarks()

## 6. Baseline runs

In [ ]:
# One CLI process per (predictor, dataset). It reloads the weights each time,
# which is the price of a guaranteed VRAM release: a 4-bit 7B leaves ~7 GB
# resident, and that is exactly the margin the next model needs on a 16 GB T4.
LIMIT = 200  # raise once a full pass fits the session budget

runs: list[str] = []
for predictor in PREDICTORS:
    for name, split in BENCHMARKS:
        if task_of(name) not in SUPPORTED[predictor]:
            continue  # change_map / fusion_multilabel belong to MCI and TerraMind
        run_id = f"baseline-{predictor}-{name}-{split}"
        sq("eval", "--dataset", name, "--split", split, "--predictor", predictor,
           "--limit", LIMIT, "--run-id", run_id, check=False)
        runs.append(run_id)

print(len(runs), "runs attempted")

## 7. Report

In [ ]:
report = RUNS / "phase1_baseline.md"
done = [str(RUNS / r) for r in runs if (RUNS / r).is_dir()]
if done:
    sq("report", "--runs", *done, "--out", str(report))

from IPython.display import Markdown, display

display(Markdown(report.read_text(encoding="utf-8") if report.exists() else "_no runs completed_"))

## What these numbers are for

`runs/phase1_baseline.md` is the **before** column. Two things to read off it
before starting the Phase 2 QLoRA run:

* Both `accuracy` and `accuracy_lenient` on the VQA rows. An untuned VLM answers
  in sentences and a tuned one answers tersely, so a strict-only delta credits
  the fine-tune for learning the *output format*. Quote both or neither.
* `acc@0.5` on VRSBench referring. Below 0.30 is the trigger — agreed in
  advance — to reopen the dedicated grounding model instead of relying on Qwen's
  native boxes.

Phase 2 re-runs exactly these commands with the adapter attached, and
`satquery compare --before runs/baseline-... --after runs/tuned-...` renders the
delta table with regressions marked. Keep the `runs/` directory: the JSON is the
artefact the report and the trace both cite.